In [1]:
from datasets import load_dataset
import numpy as np

raw_dataset = load_dataset('bentrevett/multi30k')
raw_dataset

/Users/sergheibigboy/Downloads/transfromers/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

In [2]:
for i in range(5):
    print("EN:", raw_dataset['train'][i]['en'])
    print("DE:", raw_dataset['train'][i]['de'])
    print()

EN: Two young, White males are outside near many bushes.
DE: Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.

EN: Several men in hard hats are operating a giant pulley system.
DE: Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.

EN: A little girl climbing into a wooden playhouse.
DE: Ein kleines Mädchen klettert in ein Spielhaus aus Holz.

EN: A man in a blue shirt is standing on a ladder cleaning a window.
DE: Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.

EN: Two men are at the stove preparing food.
DE: Zwei Männer stehen am Herd und bereiten Essen zu.



In [3]:
for split in raw_dataset:
    print(split, len(raw_dataset[split]))

train 29000
validation 1014
test 1000


In [4]:
the_most_large = ""
max_quant = -np.inf

for x in raw_dataset['train']['en']:
    quant = len(x.split())
    if quant > max_quant:
        the_most_large = x
        max_quant = quant

print("Longest sentence:", the_most_large)
print("Number of words:", max_quant)


Longest sentence: A soccer player wearing a red and white uniform catching the ball for a score as the opponent wearing a blue and white uniform is in the background along with a crowd of people watching the game
Number of words: 37


In [5]:
en_mean = np.mean([
    len(x.split())
    for x in raw_dataset["train"]["en"]
])

de_mean = np.mean([
    len(x.split())
    for x in raw_dataset["train"]["de"]
])

en_mean, de_mean

(np.float64(11.897241379310344), np.float64(11.116655172413793))

In [6]:
def build_vocab(sentences):
    special_tokens = ['<pad>', '<unk>', '<sos>', '<eos>']

    vocab = {}

    for word in special_tokens:
        vocab[word] = len(vocab)

    for x in sentences:
        words = x.split()
        for word in words:
            if word not in vocab.keys():
                vocab[word] = len(vocab) 
    return vocab



In [7]:
en_vocab = build_vocab(raw_dataset["train"]["en"])
de_vocab = build_vocab(raw_dataset["train"]["de"])

print(len(en_vocab))
print(len(de_vocab))

print(en_vocab["<pad>"])  # 0
print(en_vocab["<unk>"])  # 1

15460
24893
0
1


In [8]:
def encode_sentence(sentence, vocab):
    encoder = []
    encoder.append(vocab['<sos>'])

    for word in sentence.split():
        if word not in vocab.keys():
            encoder.append(vocab['<unk>'])
        else:
            encoder.append(vocab[word])
    
    encoder.append(vocab['<eos>'])
    return encoder
    

In [9]:
test = encode_sentence(
    "A completelyunknownword is playing guitar",
    en_vocab
)

print(test)

[2, 23, 1, 33, 255, 46, 3]


In [10]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, dataset, en_vocab, de_vocab):
        self.dataset = dataset
        self.en_sentences = dataset['en']
        self.de_sentences = dataset['de']
        self.en_vocab = en_vocab
        self.de_vocab = de_vocab

        self.encode_en = []
        self.encode_de = []

        for sentence in self.en_sentences:
            self.encode_en.append(encode_sentence(sentence, self.en_vocab))

        for sentence in self.de_sentences:
            self.encode_de.append(encode_sentence(sentence, self.de_vocab))

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.encode_en[idx], self.encode_de[idx]


    

In [11]:
train_dataset = TranslationDataset(
    raw_dataset["train"],
    en_vocab,
    de_vocab
)

src, tgt = train_dataset[0]

print(src)
print(tgt)
print(len(train_dataset))

[2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 3]
[2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 3]
29000


In [12]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch):
    src_sentences = [torch.tensor(src, dtype=torch.long) for src, tgt in batch]
    tgt_sentences = [torch.tensor(tgt, dtype=torch.long) for src, tgt in batch]

    src_batch = pad_sequence(src_sentences, batch_first=True, padding_value=0)
    tgt_batch = pad_sequence(tgt_sentences, batch_first=True, padding_value=0)

    return src_batch, tgt_batch



In [13]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

src, tgt = next(iter(train_loader))

print(src.shape)
print(tgt.shape)
print(src)
print(tgt)

torch.Size([32, 21])
torch.Size([32, 20])
tensor([[    2,    23,   185,   574,    33,    34,    15,    19,   167,   316,
            19,  2757,     3,     0,     0,     0,     0,     0,     0,     0,
             0],
        [    2,    23,    30,    15,    19,    90,    32,    33,   676,    19,
          2046,   261,     3,     0,     0,     0,     0,     0,     0,     0,
             0],
        [    2,    23,    63,    33,   361,   116,  6692,    35,    40,  1001,
             3,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0],
        [    2,    23,    30,    64,   217,    96,   156,   204,   116,    19,
           658,   220,    64,    19,  6578,  2135,     3,     0,     0,     0,
             0],
        [    2,    23,   185,   254,    15,    44,   169,    33,    34,  2451,
            35,    19,   170,     3,     0,     0,     0,     0,     0,     0,
             0],
        [    2,    23,    30,    85,    31,   671,    33,   140,    61,    64,
    

In [14]:
PAD_IDX = en_vocab['<pad>']

src_padding_mask = src.eq(PAD_IDX)


In [15]:
tgt_input = tgt[:, :-1]
tgt_output = tgt[:, 1:]

tgt_input.shape, tgt_output.shape

tgt_padding_mask = tgt_input.eq(PAD_IDX)

In [16]:
import import_ipynb
from transformer_from_sc import *

In [17]:
D_MODEL = 128
NUM_HEADS = 4
D_FF = 512
N_LAYERS = 2
DROPOUT = 0.2

In [20]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [21]:
encoder = TransformerEncoder(
    len(en_vocab), D_MODEL, NUM_HEADS, D_FF, N_LAYERS, DROPOUT
)
decoder = TransformerDecoder(
    len(de_vocab), D_MODEL, NUM_HEADS, D_FF, N_LAYERS, DROPOUT
)
model = Transformer(encoder, decoder).to(device)

In [23]:
src, tgt = next(iter(train_loader))

src = src.to(device)
tgt = tgt.to(device)

tgt_input = tgt[:, :-1]
tgt_output = tgt[:, 1:]

src_padding_mask = src.eq(PAD_IDX)
tgt_padding_mask = tgt_input.eq(PAD_IDX)

logits = model(
    src,
    tgt_input,
    src_padding_mask=src_padding_mask,
    tgt_padding_mask=tgt_padding_mask
)

print(logits.shape)

torch.Size([32, 24, 24893])


In [24]:
import torch.nn as nn
criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
    label_smoothing=0.1
)

logits.reshape(-1, logits.size(-1))
tgt_output.reshape(-1)

loss = criterion(
    logits.reshape(-1, logits.size(-1)),
    tgt_output.reshape(-1)
)

print(loss.item())

10.637946128845215


In [25]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

device = torch.device("mps")
model = model.to(device)

src = src.to(device)
tgt = tgt.to(device)

tgt_input = tgt[:, :-1]
tgt_output = tgt[:, 1:]

src_padding_mask = src.eq(PAD_IDX)
tgt_padding_mask = tgt_input.eq(PAD_IDX)

In [26]:
val_dataset = TranslationDataset(
    raw_dataset["validation"],
    en_vocab,
    de_vocab
)

In [27]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [28]:
src, tgt = next(iter(val_loader))
print(src.shape, tgt.shape)

torch.Size([32, 24]) torch.Size([32, 27])


In [29]:
print(val_dataset[0])
src, tgt = next(iter(val_loader))
print(src.shape, tgt.shape)

([2, 23, 262, 74, 14, 8, 1313, 1603, 1986, 19, 1192, 3], [2, 68, 132, 88, 2895, 6446, 18856, 34, 55, 1314, 3])
torch.Size([32, 24]) torch.Size([32, 27])


In [30]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for src, tgt in train_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        src_padding_mask = src.eq(PAD_IDX)
        tgt_padding_mask = tgt_input.eq(PAD_IDX)

        optimizer.zero_grad()

        logits = model(
            src,
            tgt_input,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)
        

        

In [31]:
def evaluate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in val_loader:
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            src_padding_mask = src.eq(SRC_PAD_IDX)
            tgt_padding_mask = tgt_input.eq(TGT_PAD_IDX)

            logits = model(
                src,
                tgt_input,
                src_padding_mask=src_padding_mask,
                tgt_padding_mask=tgt_padding_mask
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_output.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(val_loader)

In [32]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

SRC_PAD_IDX = en_vocab["<pad>"]
TGT_PAD_IDX = de_vocab["<pad>"]

criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model = model.to(device)

N_EPOCHS = 20

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate_epoch(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d}/{N_EPOCHS} | "
        f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f}"
    )

Epoch 01/20 | train loss: 6.0678 | val loss: 5.1445
Epoch 02/20 | train loss: 4.9641 | val loss: 4.6841
Epoch 03/20 | train loss: 4.5748 | val loss: 4.4163
Epoch 04/20 | train loss: 4.3076 | val loss: 4.2061
Epoch 05/20 | train loss: 4.0999 | val loss: 4.0675
Epoch 06/20 | train loss: 3.9287 | val loss: 3.9853
Epoch 07/20 | train loss: 3.7887 | val loss: 3.9063
Epoch 08/20 | train loss: 3.6559 | val loss: 3.8295
Epoch 09/20 | train loss: 3.5432 | val loss: 3.7926
Epoch 10/20 | train loss: 3.4393 | val loss: 3.7453
Epoch 11/20 | train loss: 3.3462 | val loss: 3.6930
Epoch 12/20 | train loss: 3.2542 | val loss: 3.6701
Epoch 13/20 | train loss: 3.1714 | val loss: 3.6455
Epoch 14/20 | train loss: 3.0903 | val loss: 3.6182
Epoch 15/20 | train loss: 3.0170 | val loss: 3.6076
Epoch 16/20 | train loss: 2.9450 | val loss: 3.5919
Epoch 17/20 | train loss: 2.8741 | val loss: 3.5773
Epoch 18/20 | train loss: 2.8114 | val loss: 3.5747
Epoch 19/20 | train loss: 2.7500 | val loss: 3.5505
Epoch 20/20 

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "en_vocab": en_vocab,
    "de_vocab": de_vocab,
}, "en_de_transformer.pt")

SRC_PAD_IDX = en_vocab["<pad>"]
TGT_PAD_IDX = de_vocab["<pad>"]
id_to_de = {idx: word for word, idx in de_vocab.items()}

@torch.inference_mode()
def translate(text, max_len=30):
    model.eval()

    src_ids = encode_sentence(text, en_vocab)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)
    src_mask = src.eq(SRC_PAD_IDX)

    encoder_output = model.encoder(src, src_padding_mask=src_mask)

    generated = torch.tensor(
        [[de_vocab["<sos>"]]],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_len):
        logits = model.decoder(
            generated,
            encoder_output,
            tgt_padding_mask=generated.eq(TGT_PAD_IDX),
            src_padding_mask=src_mask
        )

        next_id = logits[0, -1].argmax().item()

        if next_id == de_vocab["<eos>"]:
            break

        generated = torch.cat(
            [generated, torch.tensor([[next_id]], device=device)],
            dim=1
        )

    return " ".join(
        id_to_de[i]
        for i in generated[0].tolist()
        if i not in {
            de_vocab["<pad>"],
            de_vocab["<sos>"],
            de_vocab["<eos>"]
        }
    )

while True:
    text = input("English (exit для выхода): ").strip()

    if text.lower() == "exit":
        break

    print("Deutsch:", translate(text))

Модель сохранена: en_de_transformer.pt
Deutsch: Ein Feuerwehrmann fährt
Deutsch: Ein Mann fährt auf einem Fahrrad
Deutsch: Fallschirmspringen.
